# Networks and Noise

Marius Pille (Berlin Institute of Health at Charité University
Medicine)  
Leon Martin (Berlin Institute of Health at Charité University
Medicine)  
Leon Stefanovski (Charité University Medicine Berlin)

In [1]:
import bsplot
import matplotlib.pyplot as plt

from tvbo import Dynamics, Network, Noise, SimulationExperiment

bsplot.style.use("tvbo")

# From a single node to a brain network

Chapter 1 worked with a single dynamical system in isolation. The TVBO
state equation has more structure than that:

$$
dS_i = \left[f_d(S_i, \theta^d, C_i, I_i)\right]dt + g(S_i, \theta^g)\,dW_i.
$$

Two terms bring brains into the picture:

-   **Coupling** $C_i$ through a network of regions.
-   **Noise** $g(S_i, \theta^g)\,dW_i$ that turns the equation into an
    SDE.

This notebook introduces both with the native TVBO API and minimal code.

## A toy network

Networks in TVBO are first-class metadata objects. A small connectome
can be written inline as YAML; a real one is loaded from the database.

In [2]:
toy = Network.from_string("""
label: ToyChain
nodes:
  - {id: 0, label: A}
  - {id: 1, label: B}
  - {id: 2, label: C}
edges:
  - {source: 0, target: 1, parameters: {weight: {value: 0.6}, distance: {value: 30, unit: mm}}}
  - {source: 1, target: 2, parameters: {weight: {value: 0.5}, distance: {value: 40, unit: mm}}}
  - {source: 0, target: 2, parameters: {weight: {value: 0.3}, distance: {value: 60, unit: mm}}}
""")
toy.plot_graph()
plt.show()

The Desikan-Killiany connectome ships with TVBO and is loaded the same
way:

In [3]:
sc = Network.from_db(rec="avgMatrix", atlas="DesikanKilliany")
print(sc.label, "—", sc.number_of_nodes, "regions")
sc.plot_overview()
plt.show()

DesikanKilliany (avgMatrix SC+FC) — 84 regions

## Simulating a coupled network

`SimulationExperiment` accepts a `Dynamics`, a `Network`, and a coupling
kernel. We use the generic 2D oscillator coupled linearly across the toy
connectome and run with the JAX-based `tvboptim` backend.

In [4]:
model = Dynamics.from_db("Generic2dOscillator")

exp = SimulationExperiment(
    dynamics=model,
    network=toy,
    coupling="Linear",
)
exp.integration.duration = 1000      # ms
exp.integration.step_size = 0.1      # ms
exp.integration.method = "Heun"

result = exp.run("tvboptim")
result.integration.data


STEP 1: Running simulation...
  Simulation period: 1000.0 ms, dt: 0.1 ms
  Transient period: 0.0 ms
  Simulation complete.

Experiment complete.

<xarray.DataArray (time: 10000, variable: 2, node: 3)> Size: 480kB
array([[[ 0.10025272, 0.10025289, 0.10025264],
 [ 0.09380361, 0.09380361, 0.09380361]],

 [[ 0.10049334, 0.10049368, 0.10049317],
 [ 0.08761468, 0.08761467, 0.08761468]],

 [[ 0.10072186, 0.10072237, 0.1007216 ],
 [ 0.08143342, 0.0814334 , 0.08143343]],

 ...,

 [[-0.18869964, -0.18871028, -0.18869432],
 [-0.11300357, -0.11289718, -0.11305677]],

 [[-0.18869964, -0.18871028, -0.18869432],
 [-0.11300357, -0.11289718, -0.11305677]],

 [[-0.18869964, -0.18871028, -0.18869432],
 [-0.11300357, -0.11289718, -0.11305677]]], shape=(10000, 2, 3))
Coordinates:
 * time (time) float64 80kB 0.1 0.2 0.3 0.4 ... 999.7 999.8 999.9 1e+03
 * variable (variable) <U1 8B 'V' 'W'
 * node (node) <U1 12B 'A' 'B' 'C' xarray.DataArray time : 10000 variable : 2 node : 3 0.1003 0.1003 0.1003 0.0938 0.0938 ... -0.1887 -0.113 -0.1129 -0.1131 array([[[ 0.10025272, 0.10025289, 0.10025264],
 [ 0.09380361, 0.09380361, 0.09380361]],

 [[ 0.10049334, 0.10049368, 0.10049317],
 [ 0.08761468, 0.08761467, 0.08761468]],

 [[ 0.10072186, 0.10072237, 0.1007216 ],
 [ 0.08143342, 0.0814334 , 0.08143343]],

 ...,

 [[-0.18869964, -0.18871028, -0.18869432],
 [-0.11300357, -0.11289718, -0.11305677]],

 [[-0.18869964, -0.18871028, -0.18869432],
 [-0.11300357, -0.11289718, -0.11305677]],

 [[-0.18869964, -0.18871028, -0.18869432],
 [-0.11300357, -0.11289718, -0.11305677]]], shape=(10000, 2, 3)) Coordinates: (3) time (time) float64 0.1 0.2 0.3 ... 999.8 999.9 1e+03 array([1.000e-01, 2.000e-01, 3.000e-01, ..., 9.998e+02, 9.999e+02, 1.000e+03],
 shape=(10000,)) variable (variable) <U1 'V' 'W' array(['V', 'W'], dtype='<U1') node (node) <U1 'A' 'B' 'C' array(['A', 'B', 'C'], dtype='<U1')

The result is an `xarray.DataArray` with named axes. We select the first
state variable and let TVBO plot all regions.

In [5]:
fig = result.integration.plot()
fig.suptitle("Deterministic network — Generic2dOscillator on ToyChain")
plt.show()

## Adding noise

Noise enters the equation through the diffusion term
$g(S, \theta^g)\,dW$. TVBO exposes it as a metadata object that we
attach to `integration`. Two flavours are built in.

### Gaussian (white) noise

In [6]:
exp.integration.noise = Noise(
    noise_type="gaussian",
    parameters={"sigma": {"value": 0.05}},
)
result_white = exp.run("tvboptim")
fig = result_white.integration.plot()
fig.suptitle("Stochastic network — additive Gaussian noise (σ = 0.05)")
plt.show()


STEP 1: Running simulation...
  Simulation period: 1000.0 ms, dt: 0.1 ms
  Transient period: 0.0 ms
  Simulation complete.

Experiment complete.

### Ornstein–Uhlenbeck (coloured) noise

OU noise has finite correlation time `ntau` and intensity `nsig`,
producing temporally smooth fluctuations.

In [7]:
exp.integration.noise = Noise(
    noise_type="ou",
    parameters={"nsig": {"value": 0.05}, "ntau": {"value": 5.0}},
)
result_ou = exp.run("tvboptim")
fig = result_ou.integration.plot()
fig.suptitle("Stochastic network — Ornstein–Uhlenbeck noise (ntau = 5 ms)")
plt.show()


STEP 1: Running simulation...
  Simulation period: 1000.0 ms, dt: 0.1 ms
  Transient period: 0.0 ms
  Simulation complete.

Experiment complete.

## What we have so far

-   A `Network` is metadata, just like a `Dynamics`.
-   `SimulationExperiment(dynamics, network, coupling)` is the canonical
    container.
-   `integration.noise = Noise(...)` turns the deterministic ODE into an
    SDE.
-   The result is the same xarray-backed object as before, regardless of
    network size.

The next chapter takes this same experiment and starts to *vary* the
parameters systematically.